In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')



vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\stock-footage-cars-parking-and-leaving-cctv-feed.webm")

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        results = model(frame, classes=[1, 2, 3, 5, 7])

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

        cv2.imshow("video", frame)
        cv2.waitKey(40)

vid.release()
cv2.destroyAllWindows


In [7]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

while True:
    ret, frame = vid.read()
    if not ret:
        break

    results = model(frame, classes=[1, 2, 3, 5, 7])  # cars, motorbikes, buses, trucks

    for result in results:
        boxes = result.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            # Extract detected car region
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

            # Mask out extreme dark/light pixels (shadows, glass, headlights)
            sat_mask = hsv_roi[:, :, 1] > 20
            val_mask = (hsv_roi[:, :, 2] > 40) & (hsv_roi[:, :, 2] < 250)
            valid_mask = sat_mask & val_mask

            if np.count_nonzero(valid_mask) == 0:
                continue

            # Average HSV of valid pixels
            avg_hue = np.mean(hsv_roi[:, :, 0][valid_mask])
            avg_sat = np.mean(hsv_roi[:, :, 1][valid_mask])
            avg_val = np.mean(hsv_roi[:, :, 2][valid_mask])

            # Determine color
            colour = "Undefined"
            if avg_val < 40:
                colour = "BLACK"
            elif avg_sat < 40 and avg_val > 200:
                colour = "WHITE"
            elif avg_sat < 40 and 40 <= avg_val <= 200:
                colour = "GRAY"
            elif avg_hue < 5 or avg_hue >= 170:
                colour = "RED"
            elif avg_hue < 22:
                colour = "ORANGE"
            elif avg_hue < 78:
                colour = "YELLOW"
            elif avg_hue < 131:
                colour = "BLUE"
            else:
                colour = "VIOLET"

            # Draw bounding box and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, colour, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("video", frame)
    if cv2.waitKey(40) & 0xFF == 27:  # ESC to exit
        break

vid.release()
cv2.destroyAllWindows()



0: 384x640 2 cars, 52.4ms
Speed: 1.1ms preprocess, 52.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 49.6ms
Speed: 1.0ms preprocess, 49.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 67.0ms
Speed: 1.3ms preprocess, 67.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 45.7ms
Speed: 1.0ms preprocess, 45.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 51.5ms
Speed: 0.9ms preprocess, 51.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 51.8ms
Speed: 1.3ms preprocess, 51.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.2ms
Speed: 1.0ms preprocess, 44.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 53.0ms
Speed: 1.0ms preprocess, 53.0ms inference, 1.3ms postprocess

KeyboardInterrupt: 

In [6]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

while True:
    ret, frame = vid.read()
    if not ret:
        break

    results = model(frame, classes=[1, 2, 3, 5, 7])  # cars, motorbikes, buses, trucks

    for result in results:
        boxes = result.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

            # Looser masking: only remove extreme dark/bright pixels
            val = hsv_roi[:, :, 2]
            valid_mask = (val > 5) & (val < 250)

            if np.count_nonzero(valid_mask) == 0:
                continue

            # Average HSV of valid pixels
            avg_hue = np.mean(hsv_roi[:, :, 0][valid_mask])
            avg_sat = np.mean(hsv_roi[:, :, 1][valid_mask])
            avg_val = np.mean(val[valid_mask])

            # Determine color
            colour = "Undefined"
            if avg_val < 40:
                colour = "BLACK"
            elif avg_sat < 50 and avg_val > 180:
                colour = "WHITE"
            elif avg_sat < 50 and 40 <= avg_val <= 180:
                colour = "GRAY"
            elif avg_hue < 5 or avg_hue >= 170:
                colour = "RED"
            elif avg_hue < 22:
                colour = "ORANGE"
            elif avg_hue < 78:
                colour = "YELLOW"
            elif avg_hue < 131:
                colour = "BLUE"
            else:
                colour = "VIOLET"

            # Draw bounding box and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, colour, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("video", frame)
    if cv2.waitKey(40) & 0xFF == 27:  # ESC to exit
        break

vid.release()
cv2.destroyAllWindows()



0: 384x640 2 cars, 54.4ms
Speed: 3.0ms preprocess, 54.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 50.4ms
Speed: 1.0ms preprocess, 50.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 50.9ms
Speed: 1.3ms preprocess, 50.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 49.4ms
Speed: 1.0ms preprocess, 49.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.8ms
Speed: 1.0ms preprocess, 52.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.1ms
Speed: 1.0ms preprocess, 44.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.2ms
Speed: 1.2ms preprocess, 52.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.7ms
Speed: 0.9ms preprocess, 44.7ms inference, 0.9ms postprocess

In [9]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\istockphoto-1142262134-640_adpp_is.mp4")

# Define color ranges in HSV (you can adjust these based on your needs)
COLOR_RANGES = {
    'red': [(0, 50, 50), (10, 255, 255)],
    'red2': [(170, 50, 50), (180, 255, 255)],  # Red wraps around 180°
    'blue': [(100, 50, 50), (140, 255, 255)],
    'green': [(40, 50, 50), (80, 255, 255)],
    'yellow': [(20, 50, 50), (40, 255, 255)],
    'white': [(0, 0, 200), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 30)],
}

def get_dominant_color(hsv_roi):
    """Get the dominant color from HSV ROI"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Calculate average HSV values
    avg_h = np.mean(hsv_roi[:, :, 0])
    avg_s = np.mean(hsv_roi[:, :, 1])
    avg_v = np.mean(hsv_roi[:, :, 2])
    
    # Determine color based on HSV ranges
    if avg_v < 50:
        return "black"
    elif avg_s < 30 and avg_v > 200:
        return "white"
    elif (avg_h >= 0 and avg_h <= 10) or (avg_h >= 170 and avg_h <= 180):
        return "red"
    elif avg_h >= 20 and avg_h <= 40:
        return "yellow"
    elif avg_h >= 40 and avg_h <= 80:
        return "green"
    elif avg_h >= 100 and avg_h <= 140:
        return "blue"
    else:
        return "unknown"

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Extract ROI from HSV frame
                roi_hsv = hsv_frame[y1:y2, x1:x2]
                
                # Get dominant color
                color_name = get_dominant_color(roi_hsv)
                
                # Draw rectangle with color-specific bounding box
                if color_name == "red":
                    color = (0, 0, 255)  # Red
                elif color_name == "blue":
                    color = (255, 0, 0)  # Blue
                elif color_name == "green":
                    color = (0, 255, 0)  # Green
                elif color_name == "yellow":
                    color = (0, 255, 255)  # Yellow
                elif color_name == "white":
                    color = (255, 255, 255)  # White
                elif color_name == "black":
                    color = (0, 0, 0)  # Black
                else:
                    color = (128, 128, 128)  # Gray for unknown
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                # Display color information
                label = f"{color_name} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 7 cars, 54.3ms
Speed: 3.5ms preprocess, 54.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 49.1ms
Speed: 1.8ms preprocess, 49.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 46.3ms
Speed: 1.7ms preprocess, 46.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 1 truck, 47.6ms
Speed: 1.8ms preprocess, 47.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 truck, 58.7ms
Speed: 1.6ms preprocess, 58.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 50.8ms
Speed: 1.6ms preprocess, 50.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 truck, 51.5ms
Speed: 2.9ms preprocess, 51.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 truck, 52.8ms
Speed: 1.9ms preprocess, 52.8ms inference, 1.1ms postprocess

In [11]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)], 
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        # Purple/magenta range - less common for cars but possible
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 2 cars, 47.9ms
Speed: 2.0ms preprocess, 47.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 58.6ms
Speed: 1.5ms preprocess, 58.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 51.7ms
Speed: 1.1ms preprocess, 51.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 49.2ms
Speed: 1.0ms preprocess, 49.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.7ms
Speed: 2.4ms preprocess, 46.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.6ms
Speed: 1.1ms preprocess, 46.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 50.9ms
Speed: 1.2ms preprocess, 50.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.4ms
Speed: 1.5ms preprocess, 52.4ms inference, 1.0ms postprocess

KeyboardInterrupt: 

In [12]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)],
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    # Convert to numpy arrays
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    # Get indices of boxes sorted by confidence (descending)
    indices = np.argsort(confs)[::-1]
    
    keep_indices = []
    
    while len(indices) > 0:
        # Take the box with highest confidence
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        # Get the current box
        current_box = boxes[current_idx]
        
        # Remove current index from list
        indices = indices[1:]
        
        # Calculate IoU with remaining boxes
        remaining_boxes = boxes[indices]
        
        # Calculate intersection coordinates
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        # Calculate intersection area
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        
        # Calculate union area
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        # Calculate IoU
        iou = intersection_area / (union_area + 1e-6)
        
        # Keep boxes with IoU less than threshold
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        all_boxes = []
        all_confs = []
        all_classIds = []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        # Apply Non-Maximum Suppression to remove duplicate detections
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Display class name (optional)
                class_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 2 cars, 55.3ms
Speed: 2.8ms preprocess, 55.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 42.8ms
Speed: 1.2ms preprocess, 42.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 81.9ms
Speed: 3.9ms preprocess, 81.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.8ms
Speed: 1.1ms preprocess, 46.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 50.5ms
Speed: 1.0ms preprocess, 50.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.1ms
Speed: 1.3ms preprocess, 44.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 57.3ms
Speed: 1.8ms preprocess, 57.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 45.5ms
Speed: 1.3ms preprocess, 45.5ms inference, 1.3ms postprocess

KeyboardInterrupt: 

In [15]:
vid.release()
cv2.destroyAllWindows()

In [2]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Get video properties for output video
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video_with_detections2.mp4', fourcc, fps, (frame_width, frame_height))

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)],
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    # Convert to numpy arrays
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    # Get indices of boxes sorted by confidence (descending)
    indices = np.argsort(confs)[::-1]
    
    keep_indices = []
    
    while len(indices) > 0:
        # Take the box with highest confidence
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        # Get the current box
        current_box = boxes[current_idx]
        
        # Remove current index from list
        indices = indices[1:]
        
        # Calculate IoU with remaining boxes
        remaining_boxes = boxes[indices]
        
        # Calculate intersection coordinates
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        # Calculate intersection area
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        
        # Calculate union area
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        # Calculate IoU
        iou = intersection_area / (union_area + 1e-6)
        
        # Keep boxes with IoU less than threshold
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

# Counter for progress display
frame_count = 0
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        frame_count += 1
        print(f"Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")
        
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        all_boxes = []
        all_confs = []
        all_classIds = []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        # Apply Non-Maximum Suppression to remove duplicate detections
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Display class name
                class_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        # Write the frame to output video
        output_video.write(frame)
        
        # Display progress (optional)
        cv2.imshow("Processing Video - Press 'q' to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Release everything
vid.release()
output_video.release()
cv2.destroyAllWindows()

print(f"Video processing complete! Output saved as 'output_video_with_detections.avi'")

Processing frame 1/1274 (0.1%)

0: 384x640 2 cars, 95.9ms
Speed: 4.8ms preprocess, 95.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 2/1274 (0.2%)

0: 384x640 2 cars, 131.0ms
Speed: 1.9ms preprocess, 131.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 3/1274 (0.2%)

0: 384x640 2 cars, 119.1ms
Speed: 2.3ms preprocess, 119.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 4/1274 (0.3%)

0: 384x640 2 cars, 1 truck, 119.0ms
Speed: 1.9ms preprocess, 119.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 5/1274 (0.4%)

0: 384x640 2 cars, 1 truck, 121.1ms
Speed: 3.4ms preprocess, 121.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 6/1274 (0.5%)

0: 384x640 2 cars, 1 truck, 112.8ms
Speed: 2.0ms preprocess, 112.8ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 7/1274 (0.5%)

0

using histogram instead of avg

In [11]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\istockphoto-1142262134-640_adpp_is.mp4")

# Get video properties for output video
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video_with_detections2.mp4', fourcc, fps, (frame_width, frame_height))

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)],
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    # Convert to numpy arrays
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    # Get indices of boxes sorted by confidence (descending)
    indices = np.argsort(confs)[::-1]
    
    keep_indices = []
    
    while len(indices) > 0:
        # Take the box with highest confidence
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        # Get the current box
        current_box = boxes[current_idx]
        
        # Remove current index from list
        indices = indices[1:]
        
        # Calculate IoU with remaining boxes
        remaining_boxes = boxes[indices]
        
        # Calculate intersection coordinates
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        # Calculate intersection area
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        
        # Calculate union area
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        # Calculate IoU
        iou = intersection_area / (union_area + 1e-6)
        
        # Keep boxes with IoU less than threshold
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    h = np.argmax(cv2.calcHist([hsv_roi], [0], None, [180], [0,180]))
    s = np.argmax(cv2.calcHist([hsv_roi], [1], None, [180], [0,256]))
    v = np.argmax(cv2.calcHist([hsv_roi], [2], None, [180], [0,256]))

    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

# Counter for progress display
frame_count = 0
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        frame_count += 1
        print(f"Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")
        
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        all_boxes = []
        all_confs = []
        all_classIds = []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        # Apply Non-Maximum Suppression to remove duplicate detections
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Display class name
                class_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        # Write the frame to output video
        output_video.write(frame)
        
        # Display progress (optional)
        cv2.imshow("Processing Video - Press 'q' to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Release everything
vid.release()
output_video.release()
cv2.destroyAllWindows()

print(f"Video processing complete! Output saved as 'output_video_with_detections.avi'")

Processing frame 1/853 (0.1%)

0: 384x640 7 cars, 48.7ms
Speed: 2.2ms preprocess, 48.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 2/853 (0.2%)

0: 384x640 4 cars, 56.8ms
Speed: 1.9ms preprocess, 56.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 3/853 (0.4%)

0: 384x640 5 cars, 42.8ms
Speed: 1.8ms preprocess, 42.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 4/853 (0.5%)

0: 384x640 7 cars, 1 truck, 42.0ms
Speed: 2.2ms preprocess, 42.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 5/853 (0.6%)

0: 384x640 5 cars, 1 truck, 45.0ms
Speed: 1.8ms preprocess, 45.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 6/853 (0.7%)

0: 384x640 4 cars, 1 truck, 45.0ms
Speed: 2.0ms preprocess, 45.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 7/853 (0.8%)

0: 384x640 5 cars,

two dominant colours

In [10]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\\git\\learning-cv\\edi\\istockphoto-1142262134-640_adpp_is.mp4")

# Get video properties for output video
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video_with_detections2.mp4', fourcc, fps, (frame_width, frame_height))


# ---------- Color Detection Utilities ----------

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))


def get_top2_car_colors(hsv_roi, second_color_ratio=0.2):
    """Return up to 2 dominant car colors using HSV histogram with threshold"""
    if hsv_roi.size == 0:
        return ["unknown"]
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return ["unknown"]

    # Compute histograms
    h_hist = cv2.calcHist([hsv_roi], [0], None, [180], [0,180]).flatten()
    s_hist = cv2.calcHist([hsv_roi], [1], None, [256], [0,256]).flatten()
    v_hist = cv2.calcHist([hsv_roi], [2], None, [256], [0,256]).flatten()

    dominant_s = np.argmax(s_hist)
    dominant_v = np.argmax(v_hist)

    # --- Handle grayscale cases ---
    if dominant_s < 40:
        if dominant_v < 50:
            return ["black"]
        elif dominant_v > 200:
            return ["white"]
        else:
            return ["gray"]

    # --- Otherwise use Hue histogram ---
    top2_idx = h_hist.argsort()[-2:][::-1]  # top 2 bins
    peak1, peak2 = top2_idx[0], top2_idx[1]

    def map_hue_to_color(h):
        if h < 10 or h >= 160: return "red"
        elif 10 <= h < 25: return "orange"
        elif 25 <= h < 40: return "yellow"
        elif 40 <= h < 85: return "green"
        elif 85 <= h < 140: return "blue"
        elif 140 <= h < 160: return "purple"
        else: return "unknown"

    color1 = map_hue_to_color(peak1)
    colors = [color1]

    # --- Check if second color is significant ---
    if h_hist[peak2] > second_color_ratio * h_hist[peak1]:
        color2 = map_hue_to_color(peak2)
        if color2 != color1:
            colors.append(color2)

    return colors


# ---------- YOLO + Video Processing ----------

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    indices = np.argsort(confs)[::-1]
    keep_indices = []
    
    while len(indices) > 0:
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        current_box = boxes[current_idx]
        indices = indices[1:]
        
        remaining_boxes = boxes[indices]
        
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        iou = intersection_area / (union_area + 1e-6)
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]


# ---------- Main Loop ----------

frame_count = 0
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        frame_count += 1
        print(f"Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")
        
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[2, 3, 5, 7])  # Vehicles: car, motorcycle, bus, truck

        all_boxes, all_confs, all_classIds = [], [], []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get up to 2 dominant colors
                color_names = get_top2_car_colors(roi_hsv)
                display_text = " + ".join([c.upper() for c in color_names])
                
                # Pick first color for box
                bgr_color, _ = get_color_display_properties(color_names[0])
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display detected colors + confidence
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Show class name
                class_names = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        output_video.write(frame)
        cv2.imshow("Processing Video - Press 'q' to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

vid.release()
output_video.release()
cv2.destroyAllWindows()

print("Video processing complete! Output saved as 'output_video_with_detections2.mp4'")


Processing frame 1/853 (0.1%)

0: 384x640 7 cars, 52.6ms
Speed: 2.6ms preprocess, 52.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 2/853 (0.2%)

0: 384x640 4 cars, 55.0ms
Speed: 2.1ms preprocess, 55.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 3/853 (0.4%)

0: 384x640 5 cars, 45.8ms
Speed: 2.1ms preprocess, 45.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 4/853 (0.5%)

0: 384x640 7 cars, 1 truck, 45.5ms
Speed: 1.8ms preprocess, 45.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 5/853 (0.6%)

0: 384x640 5 cars, 1 truck, 47.3ms
Speed: 2.5ms preprocess, 47.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 6/853 (0.7%)

0: 384x640 4 cars, 1 truck, 47.0ms
Speed: 1.6ms preprocess, 47.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 7/853 (0.8%)

0: 384x640 5 cars,

using k-means clustering 

In [15]:
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.cluster import KMeans
import math
import time

# --------------------------------------------------
# Basic color naming support (same logic simplified)
# --------------------------------------------------
CANONICAL_COLORS = {
    "white": (255,255,255),
    "black": (0,0,0),
    "gray": (128,128,128),
    "silver": (192,192,192),
    "red": (180, 20, 20),
    "blue": (30, 70, 170),
    "green": (40,130,40),
    "yellow": (240,220,50),
    "orange": (230,140,30),
    "brown": (90,60,30),
    "purple": (120,40,140),
    "pink": (255,150,180)
}

HSV_RULES = [
    ("black",  (0,179), 0,   0,   255, 55),
    ("white",  (0,179), 0, 200,   40, 255),
    ("gray",   (0,179), 0,  60,   40, 200),
    ("red",    (0,10),  50,  50),
    ("red",    (170,179),50, 50),
    ("orange", (11,25),  70,  70),
    ("yellow", (26,35),  70,  70),
    ("green",  (36,85),  55,  50),
    ("blue",   (101,130),55, 50),
    ("purple", (131,155),45, 50),
    ("pink",   (156,169),30,160),
]

DISPLAY_COLORS = {
    'red': (0,0,255),
    'blue': (255,0,0),
    'green': (0,255,0),
    'yellow': (0,255,255),
    'orange': (0,165,255),
    'white': (255,255,255),
    'black': (0,0,0),
    'gray': (128,128,128),
    'silver': (192,192,192),
    'brown': (42,42,165),
    'purple': (128,0,128),
    'pink': (180,105,255),
    'unknown': (128,128,128)
}

CLASS_NAME_MAP = {2:'car',3:'motorcycle',5:'bus',7:'truck'}

def map_hsv_rule(h,s,v):
    for name,(hmin,hmax),smin,vmin,*rest in HSV_RULES:
        smax = rest[0] if len(rest)>0 else 255
        vmax = rest[1] if len(rest)>1 else 255
        if hmin <= h <= hmax and smin <= s <= smax and vmin <= v <= vmax:
            return name
    return None

def simple_lab_distance(rgb1, rgb2):
    # Fast approximate distance in linear-ish space (good enough for fallback)
    return np.linalg.norm(np.array(rgb1, dtype=np.float32) - np.array(rgb2, dtype=np.float32))

def lab_nearest(rgb):
    best_name = "unknown"
    best_d = 1e9
    for name, ref_rgb in CANONICAL_COLORS.items():
        d = simple_lab_distance(rgb, ref_rgb)
        if d < best_d:
            best_d = d
            best_name = name
    return best_name

def choose_final(rule_name, lab_name):
    if rule_name and lab_name:
        if rule_name == lab_name:
            return rule_name
        # simple harmonization
        pairs = {("gray","silver"),("silver","gray")}
        if (rule_name, lab_name) in pairs:
            return "silver"
        return rule_name  # prefer rule when both exist
    return rule_name or lab_name or "unknown"

def dominant_color_kmeans(roi_bgr,
                          k=3,
                          sat_thresh=25,
                          val_thresh=40,
                          max_pixels=15000,
                          sample_pixels=4000,
                          min_cluster_fraction=0.08):
    if roi_bgr.size == 0:
        return "unknown"

    h, w = roi_bgr.shape[:2]
    total = h*w
    if total > max_pixels:
        scale = math.sqrt(max_pixels/total)
        roi_bgr = cv2.resize(roi_bgr, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)

    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    H,S,V = cv2.split(hsv)
    mask = (S >= sat_thresh) & (V >= val_thresh)
    idx = np.where(mask)
    if idx[0].shape[0] < 60:
        return "unknown"

    data = hsv[idx]
    if data.shape[0] > sample_pixels:
        sel = np.random.default_rng(42).choice(data.shape[0], sample_pixels, replace=False)
        data = data[sel]

    if data.shape[0] < k:
        k = max(1, data.shape[0])

    if k <= 0:
        return "unknown"

    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    try:
        kmeans.fit(data.astype(np.float32))
    except:
        return "unknown"

    centers = kmeans.cluster_centers_
    labels = kmeans.labels_
    uniq, counts = np.unique(labels, return_counts=True)
    fractions = counts / counts.sum()

    clusters = []
    for cid, frac in zip(uniq, fractions):
        if frac < min_cluster_fraction and len(uniq) > 1:
            continue
        h_c, s_c, v_c = centers[cid]
        hsv_pixel = np.array([[[h_c, s_c, v_c]]], dtype=np.uint8)
        bgr = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2BGR)[0,0]
        r,g,b = int(bgr[2]), int(bgr[1]), int(bgr[0])
        rule = map_hsv_rule(int(h_c), int(s_c), int(v_c))
        labn = lab_nearest((r,g,b))
        final = choose_final(rule, labn)
        clusters.append((final, frac))
    if not clusters:
        return "unknown"
    clusters.sort(key=lambda x: x[1], reverse=True)
    return clusters[0][0]

# ---------------------------
# MAIN LOOP (YOUR STYLE)
# ---------------------------
if __name__ == "__main__":
    # EXACT style you requested:
    vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\istockphoto-1142262134-640_adpp_is.mp4")

    if not vid.isOpened():
        raise RuntimeError("Could not open video file")

    # Prepare output writer
    frame_width  = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = vid.get(cv2.CAP_PROP_FPS) or 30
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter("output_kmeans_colors.mp4", fourcc, fps, (frame_width, frame_height))

    model = YOLO("yolov8n.pt")  # or path to a custom model

    frame_count = 0
    total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    t0 = time.time()

    while True:
        ret, frame = vid.read()
        if not ret:
            break
        frame_count += 1

        results = model(frame, classes=[2,3,5,7], conf=0.25, verbose=False)  # vehicles
        for r in results:
            if r.boxes is None:
                continue
            boxes = r.boxes.xyxy.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            cls_ids = r.boxes.cls.cpu().numpy().astype(int)

            for (x1,y1,x2,y2), conf, cls_id in zip(boxes, confs, cls_ids):
                x1,y1,x2,y2 = map(int, (x1,y1,x2,y2))
                x1 = max(0,x1); y1=max(0,y1); x2=min(frame.shape[1]-1,x2); y2=min(frame.shape[0]-1,y2)
                if x2-x1 < 12 or y2-y1 < 12:
                    continue
                roi = frame[y1:y2, x1:x2]

                color_name = dominant_color_kmeans(roi, k=3)
                bgr_color = DISPLAY_COLORS.get(color_name, (128,128,128))

                cv2.rectangle(frame, (x1,y1), (x2,y2), bgr_color, 2)
                label1 = f"{color_name.upper()}"
                label2 = f"{CLASS_NAME_MAP.get(cls_id,'veh')} {conf:.2f}"

                # Draw labels
                def draw(bg_y, text):
                    (tw,th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                    cv2.rectangle(frame, (x1, bg_y - th - 6), (x1 + tw + 4, bg_y), bgr_color, cv2.FILLED)
                    fg = (0,0,0) if sum(bgr_color)//3 > 150 else (255,255,255)
                    cv2.putText(frame, text, (x1+2, bg_y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, fg, 1, cv2.LINE_AA)

                draw(y1, label1)
                draw(y1 - 24, label2)

        out.write(frame)
        cv2.imshow("Vehicle Color (KMeans)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        if frame_count % 30 == 0 and total_frames > 0:
            elapsed = time.time() - t0
            print(f"Frame {frame_count}/{total_frames}  {(frame_count/total_frames*100):.1f}%  ProcFPS={(frame_count/elapsed):.2f}")

    vid.release()
    out.release()
    cv2.destroyAllWindows()
    print("Done. Saved to output_kmeans_colors.mp4")

Frame 30/853  3.5%  ProcFPS=10.69
Frame 60/853  7.0%  ProcFPS=9.92
Frame 90/853  10.6%  ProcFPS=10.09
Frame 120/853  14.1%  ProcFPS=10.31
Frame 150/853  17.6%  ProcFPS=10.50
Frame 180/853  21.1%  ProcFPS=10.45
Frame 210/853  24.6%  ProcFPS=10.37
Frame 240/853  28.1%  ProcFPS=10.36
Frame 270/853  31.7%  ProcFPS=10.40
Frame 300/853  35.2%  ProcFPS=10.43
Frame 330/853  38.7%  ProcFPS=10.53
Frame 360/853  42.2%  ProcFPS=10.63
Frame 390/853  45.7%  ProcFPS=10.69
Frame 420/853  49.2%  ProcFPS=10.66
Frame 450/853  52.8%  ProcFPS=10.71
Frame 480/853  56.3%  ProcFPS=10.69
Frame 510/853  59.8%  ProcFPS=10.65
Frame 540/853  63.3%  ProcFPS=10.67
Frame 570/853  66.8%  ProcFPS=10.69
Frame 600/853  70.3%  ProcFPS=10.75
Frame 630/853  73.9%  ProcFPS=10.74
Frame 660/853  77.4%  ProcFPS=10.72
Frame 690/853  80.9%  ProcFPS=10.67
Frame 720/853  84.4%  ProcFPS=10.67
Frame 750/853  87.9%  ProcFPS=10.65
Frame 780/853  91.4%  ProcFPS=10.64
Frame 810/853  95.0%  ProcFPS=10.60
Frame 840/853  98.5%  ProcFPS=10.6

In [14]:
#!/usr/bin/env python3
"""
Video Vehicle Color Detection via YOLO + HSV K-Means

Features:
- YOLOv8 detection (or tracking) of vehicles
- Dominant paint color estimation per vehicle ROI using K-Means in HSV
- HSV filtering to avoid windows/shadows
- Rule-based + Lab fallback color naming
- Optional temporal smoothing per track id
- Efficient sampling for speed

Dependencies:
  pip install ultralytics opencv-python scikit-learn numpy scikit-image

Usage (detection per frame):
  python car_color_video_kmeans.py --video input.mp4 --output output.mp4

Usage (with tracker & smoothing):
  python car_color_video_kmeans.py --video input.mp4 --output output.mp4 --track

Author: You
"""

from __future__ import annotations
import argparse
import cv2
import numpy as np
from sklearn.cluster import KMeans
from collections import defaultdict, deque
import math
import time
import os

try:
    from skimage import color as skcolor
    _HAS_SKIMAGE = True
except Exception:
    _HAS_SKIMAGE = False

from ultralytics import YOLO

# -----------------------------
# Color Reference & Rules
# -----------------------------
CANONICAL_COLORS = {
    "white":    (255, 255, 255),
    "black":    (0, 0, 0),
    "gray":     (128, 128, 128),
    "silver":   (192, 192, 192),
    "red":      (180, 20, 20),
    "maroon":   (90, 10, 10),
    "blue":     (30, 70, 170),
    "navy":     (15, 25, 70),
    "light_blue": (120, 180, 255),
    "green":    (40, 130, 40),
    "dark_green": (15, 60, 15),
    "yellow":   (240, 220, 50),
    "gold":     (210, 180, 60),
    "orange":   (230, 140, 30),
    "brown":    (90, 60, 30),
    "beige":    (215, 205, 175),
    "purple":   (120, 40, 140),
    "violet":   (160, 80, 200),
    "pink":     (255, 150, 180),
}

HSV_RULES = [
    ("black",  (0,179), 0,   0,   255, 55),
    ("white",  (0,179), 0, 200,   40, 255),
    ("gray",   (0,179), 0,  60,   40, 200),
    ("red",    (0, 10), 50,  50),
    ("red",    (170,179),50, 50),
    ("orange", (11, 25), 70,  70),
    ("yellow", (26, 35), 70,  70),
    ("green",  (36, 85), 55,  50),
    ("cyan",   (86,100), 50,  50),
    ("blue",   (101,130),55, 50),
    ("purple", (131,155),45, 50),
    ("pink",   (156,169),30,160),
]

PREFERRED_NAMES = {"white","black","gray","silver","red","blue","green","yellow","orange","brown","purple"}

CLASS_NAME_MAP = {0:'person',1:'bicycle',2:'car',3:'motorcycle',5:'bus',7:'truck'}

# -----------------------------
# Utility Functions
# -----------------------------
def bgr_to_rgb(bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def rgb_to_lab(rgb_arr: np.ndarray) -> np.ndarray:
    arr = rgb_arr.astype(np.float32)/255.0
    if _HAS_SKIMAGE:
        return skcolor.rgb2lab(arr.reshape(-1,1,1,3)).reshape(-1,3)
    # Manual approximate conversion
    def srgb_to_lin(c):
        return np.where(c<=0.04045, c/12.92, ((c+0.055)/1.055)**2.4)
    lin = srgb_to_lin(arr)
    M = np.array([[0.4124564,0.3575761,0.1804375],
                  [0.2126729,0.7151522,0.0721750],
                  [0.0193339,0.1191920,0.9503041]], dtype=np.float32)
    XYZ = lin @ M.T
    Xn,Yn,Zn = 0.95047,1.0,1.08883
    x,y,z = XYZ[:,0]/Xn, XYZ[:,1]/Yn, XYZ[:,2]/Zn
    delta = 6/29
    def f(t): return np.where(t>delta**3, np.cbrt(t), t/(3*delta**2)+4/29)
    fx,fy,fz = f(x),f(y),f(z)
    L = 116*fy - 16
    a = 500*(fx - fy)
    b = 200*(fy - fz)
    return np.stack([L,a,b], axis=1)

def delta_e_cie76(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.linalg.norm(a-b, axis=-1)

def map_hsv_rule(h, s, v):
    for rule in HSV_RULES:
        name, (hmin,hmax), smin, vmin, *rest = rule
        smax = rest[0] if len(rest)>0 else 255
        vmax = rest[1] if len(rest)>1 else 255
        if hmin <= h <= hmax and smin <= s <= smax and vmin <= v <= vmax:
            return name
    return None

def lab_nearest_color(rgb):
    if not hasattr(lab_nearest_color, "_cache"):
        keys = list(CANONICAL_COLORS.keys())
        vals = np.array([CANONICAL_COLORS[k] for k in keys], dtype=np.uint8)
        labs = rgb_to_lab(vals)
        lab_nearest_color._cache = (keys, labs)
    keys, labs = lab_nearest_color._cache
    target = rgb_to_lab(np.array([rgb], dtype=np.uint8))
    d = delta_e_cie76(labs, target)
    return keys[int(np.argmin(d))]

def choose_final(rule_name, lab_name):
    if rule_name and lab_name:
        if rule_name == lab_name:
            return rule_name
        if rule_name in PREFERRED_NAMES and lab_name not in PREFERRED_NAMES:
            return rule_name
        # Minor reconciliation: merge close semantics
        synonyms = {
            ("gray","silver"): "silver",
            ("silver","gray"): "silver",
            ("navy","blue"): "blue",
            ("light_blue","blue"): "blue",
            ("maroon","red"): "red",
            ("violet","purple"): "purple",
        }
        if (rule_name, lab_name) in synonyms:
            return synonyms[(rule_name, lab_name)]
        return lab_name
    return rule_name or lab_name or "unknown"

# -----------------------------
# Dominant Color via KMeans
# -----------------------------
def extract_roi_dominant_color(
    roi_bgr: np.ndarray,
    k: int = 3,
    sat_thresh: int = 25,
    val_thresh: int = 40,
    max_pixels: int = 15000,
    sample_pixels: int = 4000,
    min_cluster_fraction: float = 0.08
):
    """
    Returns dict with:
      name, hsv_center, rgb_center, confidence (fraction), clusters
    Fallback to 'unknown' if insufficient data.
    """
    if roi_bgr.size == 0 or roi_bgr.shape[0] < 6 or roi_bgr.shape[1] < 6:
        return {"name":"unknown"}

    # Optional downscale large ROIs to cap pixel count
    h, w = roi_bgr.shape[:2]
    total = h*w
    if total > max_pixels:
        scale = math.sqrt(max_pixels / total)
        new_w = max(10, int(w*scale))
        new_h = max(10, int(h*scale))
        roi_bgr = cv2.resize(roi_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)

    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    H,S,V = cv2.split(hsv)

    mask = (S >= sat_thresh) & (V >= val_thresh)
    valid = np.where(mask)
    valid_count = valid[0].shape[0]
    if valid_count < 50:
        # try relax thresholds if car is grayish
        mask_relaxed = (V > 50)
        valid = np.where(mask_relaxed)
        valid_count = valid[0].shape[0]
        if valid_count < 50:
            return {"name": "unknown"}

    pixels = hsv[valid]  # (N,3)
    # Sample if needed
    if pixels.shape[0] > sample_pixels:
        rng = np.random.default_rng(123)
        idx = rng.choice(pixels.shape[0], size=sample_pixels, replace=False)
        pixels_sample = pixels[idx]
    else:
        pixels_sample = pixels

    k_eff = min(k, pixels_sample.shape[0])
    if k_eff <= 0:
        return {"name":"unknown"}

    # Use raw HSV; optionally weight H channel less/more – here leave as is.
    feats = pixels_sample.astype(np.float32)

    try:
        km = KMeans(n_clusters=k_eff, random_state=42, n_init="auto")
        km.fit(feats)
    except Exception:
        return {"name":"unknown"}

    centers = km.cluster_centers_
    labels = km.labels_
    uniq, counts = np.unique(labels, return_counts=True)
    fractions = counts / counts.sum()

    cluster_info = []
    for cid, frac in zip(uniq, fractions):
        if frac < min_cluster_fraction and k_eff > 1:
            continue
        h_c, s_c, v_c = centers[cid]
        hsv_pixel = np.array([[[h_c, s_c, v_c]]], dtype=np.uint8)
        bgr_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2BGR)[0,0,:]
        rgb = tuple(int(x) for x in bgr_pixel[::-1])

        rule_name = map_hsv_rule(int(round(h_c)), int(round(s_c)), int(round(v_c)))
        lab_name = lab_nearest_color(rgb)
        final = choose_final(rule_name, lab_name)

        cluster_info.append({
            "fraction": float(frac),
            "hsv_center": (float(h_c), float(s_c), float(v_c)),
            "rgb_center": rgb,
            "rule_name": rule_name,
            "lab_name": lab_name,
            "final_name": final
        })

    if not cluster_info:
        return {"name":"unknown"}

    cluster_info.sort(key=lambda c: c["fraction"], reverse=True)
    dominant = cluster_info[0]
    return {
        "name": dominant["final_name"],
        "dominant_fraction": dominant["fraction"],
        "dominant_hsv": dominant["hsv_center"],
        "dominant_rgb": dominant["rgb_center"],
        "clusters": cluster_info,
        "valid_pixels": int(pixels_sample.shape[0])
    }

# -----------------------------
# Temporal Smoothing
# -----------------------------
class ColorSmoother:
    """
    Maintains a short history of color names per track id and chooses a stable output.
    """
    def __init__(self, max_history=8):
        self.history = defaultdict(lambda: deque(maxlen=max_history))

    def update(self, track_id, color_name):
        self.history[track_id].append(color_name)
        return self.get(track_id)

    def get(self, track_id):
        hist = self.history[track_id]
        if not hist:
            return "unknown"
        # Majority vote
        vals, counts = np.unique(hist, return_counts=True)
        return vals[int(np.argmax(counts))]

# -----------------------------
# Drawing Helpers
# -----------------------------
COLOR_DISPLAY_MAP = {
    'red': (0,0,255),
    'blue': (255,0,0),
    'green': (0,255,0),
    'yellow': (0,255,255),
    'orange': (0,165,255),
    'white': (255,255,255),
    'black': (0,0,0),
    'gray': (128,128,128),
    'silver': (192,192,192),
    'brown': (42,42,165),
    'purple': (128,0,128),
    'pink': (180,105,255),
    'beige': (200,220,255),
    'maroon': (0,0,128),
    'navy': (128,0,0),
    'light_blue': (255,200,100),
    'gold': (0,215,255),
    'violet': (238,130,238),
    'unknown': (128,128,128)
}

def draw_label(img, x1, y1, text, color, font_scale=0.5, thickness=1):
    # Draw filled background for readability
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, cv2.FILLED)
    # Put contrasting text (black or white)
    avg = sum(color)//3
    text_color = (0,0,0) if avg > 150 else (255,255,255)
    cv2.putText(img, text, (x1+2, y1-4), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, thickness, cv2.LINE_AA)

# -----------------------------
# Main Processing Loop
# -----------------------------
def process_video(
    video_path: str,
    output_path: str,
    model_path: str = "yolov8n.pt",
    k: int = 3,
    track: bool = False,
    conf: float = 0.25,
    device: str | None = None,
    smooth: bool = True,
    show: bool = True,
    skip_if_low_conf: float = 0.15,
    frame_stride: int = 1
):
    """
    track: if True uses model.track for persistent ids
    frame_stride: process every Nth frame (others just copy) for speed
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(video_path)

    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Could not open video")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height= int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width,height))

    smoother = ColorSmoother() if smooth else None
    color_cache = {}  # (quantized_center)->name to reduce repeated naming overhead
    frame_idx = 0
    t0 = time.time()

    print(f"Starting processing: {video_path}")
    print(f"Frames: {total_frames}, FPS: {fps:.2f}, Size: {width}x{height}")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            if frame_stride > 1 and frame_idx % frame_stride != 0:
                # Just write original frame to keep smooth playback
                out.write(frame)
                if show:
                    cv2.imshow("Car Color Detection", frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
                continue

            # Run detection or tracking
            if track:
                results = model.track(frame, persist=True, conf=conf, verbose=False, classes=[2,3,5,7], device=device)
            else:
                results = model(frame, conf=conf, verbose=False, classes=[2,3,5,7], device=device)

            # Each result corresponds to one image (frame)
            for r in results:
                if r.boxes is None or r.boxes.xyxy is None:
                    continue
                boxes = r.boxes.xyxy.cpu().numpy()
                confs = r.boxes.conf.cpu().numpy()
                cls_ids = r.boxes.cls.cpu().numpy().astype(int)
                ids = None
                if track and hasattr(r.boxes, "id") and r.boxes.id is not None:
                    ids = r.boxes.id.cpu().numpy().astype(int)

                for i,(box, score, cls_id) in enumerate(zip(boxes, confs, cls_ids)):
                    if score < skip_if_low_conf:
                        continue
                    x1,y1,x2,y2 = map(int, box)
                    # Clip to frame
                    x1 = max(0,x1); y1=max(0,y1); x2=min(width-1,x2); y2=min(height-1,y2)
                    if x2 - x1 < 12 or y2 - y1 < 12:
                        continue

                    roi = frame[y1:y2, x1:x2]

                    # Dominant color via KMeans
                    result = extract_roi_dominant_color(roi, k=k)

                    color_name = result.get("name","unknown")

                    # Optionally smooth by track id
                    track_id = None
                    if ids is not None:
                        track_id = ids[i]
                        if smoother and track_id is not None:
                            color_name = smoother.update(track_id, color_name)

                    bgr_color = COLOR_DISPLAY_MAP.get(color_name, (128,128,128))
                    cv2.rectangle(frame, (x1,y1), (x2,y2), bgr_color, 2)

                    label_main = f"{color_name.upper()}"
                    if track_id is not None:
                        label_main += f" ID:{track_id}"
                    label_aux = f"{CLASS_NAME_MAP.get(cls_id,'veh')} {score:.2f}"
                    draw_label(frame, x1, y1, label_main, bgr_color)
                    draw_label(frame, x1, y1-22, label_aux, bgr_color)

            out.write(frame)

            if show:
                cv2.imshow("Car Color Detection", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            if frame_idx % 30 == 0:
                elapsed = time.time() - t0
                fps_proc = frame_idx / elapsed if elapsed > 0 else 0
                print(f"[{frame_idx}/{total_frames}] {fps_proc:.2f} FPS (processing)")

    finally:
        cap.release()
        out.release()
        if show:
            cv2.destroyAllWindows()

    print(f"Done. Saved to {output_path}")

# -----------------------------
# CLI
# -----------------------------
def parse_args():
    ap = argparse.ArgumentParser(description="YOLO Vehicle Color Detection (KMeans HSV)")
    ap.add_argument("--video", required=True, help="Input video path")
    ap.add_argument("--output", default="output_color_vehicles.mp4", help="Output video path")
    ap.add_argument("--model", default="yolov8n.pt", help="YOLO model path or name")
    ap.add_argument("--k", type=int, default=3, help="K for KMeans per ROI")
    ap.add_argument("--conf", type=float, default=0.25, help="Detection confidence threshold")
    ap.add_argument("--track", action="store_true", help="Enable YOLO tracking for stable IDs")
    ap.add_argument("--no-show", action="store_true", help="Do not display window")
    ap.add_argument("--no-smooth", action="store_true", help="Disable temporal smoothing of color")
    ap.add_argument("--device", default=None, help="CUDA device like '0' or 'cpu'")
    ap.add_argument("--frame-stride", type=int, default=1, help="Process every Nth frame")
    return ap.parse_args()

def main():
    args = parse_args()
    process_video(
        video_path=args.video,
        output_path=args.output,
        model_path=args.model,
        k=args.k,
        track=args.track,
        conf=args.conf,
        device=args.device,
        smooth=not args.no_smooth,
        show=not args.no_show,
        frame_stride=args.frame_stride
    )

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --video VIDEO [--output OUTPUT]
                             [--model MODEL] [--k K] [--conf CONF] [--track]
                             [--no-show] [--no-smooth] [--device DEVICE]
                             [--frame-stride FRAME_STRIDE]
ipykernel_launcher.py: error: argument --frame-stride: invalid int value: 'c:\\Users\\shard\\AppData\\Roaming\\jupyter\\runtime\\kernel-v364be541a97e7f564fb4a23540f6d36ef54d0dc46.json'


SystemExit: 2

c:\Users\shard\AppData\Local\Programs\Python\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
